# ErrorInputs — malformed inputs in 2D and 3D

`MagneticStandalone` · ASN 28 negative (diagnostic) test case (TRUST-LiMe).


## Introduction

Negative (diagnostic) cases of the 2D and 3D **standalone magnetic (induction) equation** added by TRUST-LiMe.

## What it validates

This notebook instantiates one shared `src/error_inputs.data` template for both dimensions; each generated malformed-input case must
terminate with a non-zero exit code and a readable diagnostic, rather than
crashing silently:

- 2D and 3D `error_missing_eta` — a missing magnetic diffusivity is rejected;
- 2D and 3D `error_bad_nbcomposantes` — an invalid component count is rejected;
- 2D and 3D `error_bad_bc` — an invalid boundary condition is rejected;
- 2D `error_incompatible_flux` — `correction_divergence` rejects nonzero net magnetic boundary flux;

Each case is run **directly** with the `trust` command — not through
`runCases()`, which would abort the notebook on the non-zero exit.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 11 | malformed input fails cleanly | non-zero exit + readable message |

No mesh-convergence scan for this form: the diagnostic abort is mesh-independent.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Run


In [ ]:
from pathlib import Path
from string import Template
import subprocess, tempfile, shutil

TEMPLATE = Template(Path("src/error_inputs.data").read_text())
DIAGNOSTICS = {
    "missing_eta": "the keyword 'diffusivite_magnetique' (magnetic diffusivity eta) is required",
    "bad_nbcomposantes": "nb_composantes must be between 1 and the space dimension.",
    "bad_bc": "condition_qui_nexiste_pas is not a recognized keyword.",
    "flux": "Gauss law violated by boundary conditions",
}
CASES = [
    (2, "error_missing_eta", "missing_eta", None),
    (2, "error_bad_nbcomposantes", "bad_nbcomposantes", None),
    (2, "error_bad_bc", "bad_bc", None),
    (3, "error_missing_eta_3d", "missing_eta", None),
    (3, "error_bad_nbcomposantes_3d", "bad_nbcomposantes", None),
    (3, "error_bad_bc_3d", "bad_bc", None),
    (2, "error_incompatible_flux", "flux", None),
]

def mesh_code(dim, nodes):
    if dim == 2:
        nx, ny = nodes
        return f"""Mailler dom
{{
    Pave pavé
    {{
        Origine              0.  0.
        Nombre_de_Noeuds     {nx} {ny}
        Longueurs            1.  1.
    }}
    {{
        Bord gauche   X = 0.    0. <= Y <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.
    }}
}}"""
    nx, ny, nz = nodes
    return f"""Mailler dom
{{
    Pave cube
    {{
        Origine              0.  0.  0.
        Nombre_de_Noeuds     {nx} {ny} {nz}
        Longueurs            1.  1.  1.
    }}
    {{
        Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.
        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1.
    }}
}}
Polyedriser dom"""

def test_config(dim, kind):
    if dim == 2:
        nodes, edges, probe = ((17, 17) if kind == "flux" else (100, 100)), ("gauche", "droite", "bas", "haut"), "0.5 0.5"
    else:
        nodes, edges, probe = (9, 9, 9), ("gauche", "droite", "bas", "haut", "devant", "derriere"), "0.5 0.5 0.5"
    bcs = [f"            {edge:<9} Neumann_Paroi_adiabatique" for edge in edges]
    nbcomp_line = "    nb_composantes 1"
    eta, initial, correction, tmax, dt_max = "        diffusivite_magnetique Champ_Uniforme 1 1.0", "Champ_Uniforme 1 3.0", "", "1.", "0.05"
    if kind == "bad_bc":
        bcs[0] = "            gauche condition_qui_nexiste_pas"
    elif kind == "bad_nbcomposantes":
        nbcomp_line = "    nb_composantes 7"
    elif kind == "missing_eta":
        eta = "        /* diffusivite_magnetique intentionally missing */"
    elif kind == "flux":
        nbcomp_line = "    nb_composantes 2"
        eta, initial = "        diffusivite_magnetique Champ_Uniforme 1 0.05", "Champ_Fonc_xyz dom 2 x 0."
        correction, tmax, dt_max = "        correction_divergence { solveur_divergence Petsc Gmres { precond diag { } } }", "0.1", "0.01"
    return {"dim": str(dim), "mesh_block": mesh_code(dim, nodes), "tmax": tmax, "dt_max": dt_max, "nbcomp_line": nbcomp_line, "correction_block": correction, "eta_line": eta,
            "boundary_lines": "\n".join(bcs), "initial_field": initial, "probe": probe}

results = []
for dim, CASE, kind, nprocs in CASES:
    config = test_config(dim, kind)
    generated = TEMPLATE.substitute(config)
    tmp = tempfile.mkdtemp(prefix=f"lime_{CASE}_")
    try:
        Path(tmp, f"{CASE}.data").write_text(generated)
        command = ["trust", CASE] + ([nprocs] if nprocs else [])
        p = subprocess.run(command, cwd=tmp, capture_output=True, text=True, timeout=300)
        rc, out = p.returncode, p.stdout + p.stderr
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    assert rc != 0, f"{CASE}: expected a non-zero exit, got {rc}"
    diagnostic = DIAGNOSTICS[kind]
    assert diagnostic in out, f"{CASE}: expected {diagnostic!r}, got:\n{out}"
    print(f"{dim}D/{CASE}: rc={rc} | {diagnostic}")
    print("OK: expected diagnostic emitted.")
    results.append((f"{dim}D/{CASE}", rc))


## Result


In [ ]:
# Negative (diagnostic) cases: run directly with `trust`, so there is no
# trustutils suite and no performance table. Report the outcome instead.
print("Negative cases complete: all malformed inputs rejected with a non-zero exit.")
for CASE, rc in results:
    print(f"case: {CASE}  rc={rc}")
